# Technische Einrichtung prüfen

**Data Engineering · Webscraping · Etappe 1**

[Startseite](../README.md) · [Testanleitung](../docs/TESTANLEITUNG_ETAPPE1.md) · [Probleme lösen](../docs/TROUBLESHOOTING.md)

Dieses Notebook prüft Deine Umgebung mit **drei frei erfundenen Büchern**. Du benötigst die installierte Workshop-Umgebung und den gestarteten Selenium-Dienst. Der Standardtest ruft keine externe Website auf.

1. Wähle den Kernel **Python (webscraping-workshop)**.
2. Lokal: Starte im Repo-Terminal `docker compose up -d`. In Codespaces startet der Nebendienst automatisch.
3. Starte den Kernel neu und führe **alle Zellen von oben nach unten** aus.

Erwartet werden am Schluss **SETUP OK**, drei gespeicherte CSV-Datensätze und eine echte, wieder beendete Browsersitzung. Ein fehlender Selenium-Dienst wird als Fehler angezeigt.


## 1. Gemeinsame Prüflogik laden

Das Notebook verwendet dieselben Funktionen wie `python scripts/verify_setup.py`. Es installiert keine Pakete. Die Repo-Wurzel wird relativ zum aktuellen Arbeitsordner gefunden; Du musst keinen persönlichen Dateipfad eintragen.


In [ ]:
from pathlib import Path
import sys

# Bootstrap: Das Notebook kann aus der Repo-Wurzel oder aus notebooks/ starten.
current = Path.cwd().resolve()
repo_candidates = [current, *current.parents]
repo_root = next(
    (path for path in repo_candidates if (path / "src/webscraping_workshop/checks.py").is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError("Bitte öffne dieses Notebook innerhalb des vollständig entpackten Repos.")
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from webscraping_workshop.checks import (
    find_repo_root, format_result, run_setup_checks, summary_text, write_report,
)

repo_root = find_repo_root(repo_root)
print("Gemeinsame Prüflogik geladen.")


## 2. Einrichtung prüfen

Die Grundprüfung führt diese Schritte aus:

- Python 3.12, Paketimporte und relative Dateipfade prüfen.
- JSON und HTML über einen kurzlebigen lokalen HTTP-Server abrufen.
- Drei HTML-Datensätze mit BeautifulSoup und dem lxml-Parser auslesen und mit JSON vergleichen.
- Eine CSV-Datei schreiben, wieder einlesen und die Werte vergleichen.
- Einen echten Selenium-Remote-Browser starten und explizit warten, bis JavaScript drei Tabellenzeilen erzeugt hat. Anschliessend wird die Sitzung beendet.

Der Browser bekommt die synthetische Seite als `data:`-URL. Er muss deshalb nicht auf `localhost` Deines Rechners zugreifen. Falls Selenium noch startet, wartet der Test bis zu 45 Sekunden auf Bereitschaft; ein Fehler danach bleibt ein Fehler.

**Für den vollständigen Test beide Einstellungen unverändert lassen.** `BROWSER_PRUEFEN = False` erlaubt bei der Fehlersuche einen klar gekennzeichneten Teiltest. `LIVE_QUELLEN_PRUEFEN = True` ergänzt getrennte HTTP- und Inhaltsprüfungen von ECB und Books to Scrape; dafür ist Internetzugriff nötig.


In [ ]:
BROWSER_PRUEFEN = True
LIVE_QUELLEN_PRUEFEN = False

report = run_setup_checks(
    repo_root,
    include_browser=BROWSER_PRUEFEN,
    online=LIVE_QUELLEN_PRUEFEN,
    on_result=lambda result: print(format_result(result), flush=True),
)
report_path = write_report(report, repo_root / "data/output/setup_report.json")
print("\nReport gespeichert: data/output/setup_report.json")


## 3. Ergebnisse ansehen

Die Tabelle unterscheidet bestandene, fehlgeschlagene und ausgelassene Prüfungen. Die Beispiel-CSV wird nur angezeigt, wenn der HTTP-/CSV-Test **in diesem Lauf** bestanden wurde. So erscheint kein alter Datenstand versehentlich als aktuelles Testergebnis.


In [ ]:
import pandas as pd
from IPython.display import display

display(pd.DataFrame(report["checks"])[["label", "status", "detail"]])
if any(check["key"] == "http_csv" and check["status"] == "pass" for check in report["checks"]):
    records = pd.read_csv(repo_root / "data/output/setup_records.csv", encoding="utf-8")
    print("Synthetische Daten aus data/output/setup_records.csv:")
    display(records)


## 4. Abschluss und nächster Schritt

- **SETUP OK:** Alle Grundprüfungen inklusive Browser waren erfolgreich.
- **TEILTEST OK:** Python, HTTP und Dateiverarbeitung funktionieren; der Browser wurde ausdrücklich ausgelassen und ist damit noch nicht bestätigt.
- **SETUP FEHLGESCHLAGEN:** Prüfe den gemeldeten Schritt anhand der [Fehlerhilfe](../docs/TROUBLESHOOTING.md). Starte nach einer Korrektur den Kernel neu und führe alle Zellen nochmals aus.
- Ein zusätzlicher **LIVE-TEST FEHLGESCHLAGEN** betrifft die Live-Quellen; deren Status wird vom Grundtest getrennt ausgewiesen.

Die nächste Zelle löst bei Fehlern absichtlich eine verständliche Exception aus. Für die Rückmeldung genügen die Abschlusszeile und der fehlgeschlagene Prüfpunkt bzw. ein Screenshot. Der JSON-Report enthält keine Selenium-Adresse oder Umgebungsvariablen.


In [ ]:
result_text = summary_text(report)
print(result_text)
if report["overall_status"] != "pass":
    raise RuntimeError(result_text + " – bitte den oben genannten Prüfpunkt korrigieren.")


[↑ Startseite](../README.md) · [Testanleitung Etappe 1](../docs/TESTANLEITUNG_ETAPPE1.md)
